# 延伸：用 Transformer 當 CPU —— 神經網路能「執行程式」嗎？

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/A1-Neural-CPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_cpu_program.py', 'q_cpu_small_model.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

LLM 能寫程式，但它能不能**執行**程式？這個單元我們設計一顆 8-bit 的迷你 CPU（風格類似 Z80/8080），然後訓練一個 Transformer 來「扮演」這顆 CPU：

* 輸入：整個程式 (ROM) + CPU 目前的狀態（暫存器、旗標、PC、記憶體）
* 輸出：執行一個指令之後的狀態

Transformer 必須自己完成 **fetch**（用 PC 找到要執行的指令，這正好是 attention 擅長的「查表」）、**decode**、**execute**（包含 8-bit 加減法與進位）。

最後我們會評估：它能跑多準？能跑多久不出錯？能不能拿來跑紅白機 (FC/NES) 或 Apple II 模擬器？

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

## 迷你 CPU：TinyCPU

* 4 個 8-bit 暫存器 `A B C D`，旗標 `Z`（結果為零）與 `CF`（進位/借位）
* 程式存在獨立的 ROM，共 16 個指令，`PC` 是 0~15
* 8 bytes 的資料記憶體 `M0`~`M7`
* 指令集（`x`, `y` 是暫存器，`n` 是 8-bit 立即值）：

| 指令 | 動作 | 指令 | 動作 |
|---|---|---|---|
| `NOP` | 無 | `AND y` / `OR y` / `XOR y` | `A = A op y`，設定 Z，CF=0 |
| `MOV x,y` | `x = y` | `LD x,[n]` | `x = M[n]` |
| `MVI x,n` | `x = n` | `ST [n],x` | `M[n] = x` |
| `ADD y` | `A = A + y`，設定 Z、CF | `JP n` | 跳到 n |
| `SUB y` | `A = A - y`，設定 Z、CF | `JZ n` / `JNZ n` / `JC n` | 條件跳躍 |
| `INC x` / `DEC x` | `x ± 1`，設定 Z | `HALT` | 停止（狀態不變） |

狀態是 15 個數字：`[PC, A, B, C, D, Z, CF, M0..M7]`；每個指令是 4 個數字：`[op, x, y, n]`。

In [ ]:
OPS = ["NOP", "MOV", "MVI", "ADD", "SUB", "INC", "DEC", "AND", "OR", "XOR",
       "LD", "ST", "JP", "JZ", "JNZ", "JC", "HALT"]
OP = {name: i for i, name in enumerate(OPS)}
REGS = "ABCD"
FIELDS = ["PC", "A", "B", "C", "D", "Z", "CF"] + [f"M{i}" for i in range(8)]
N_OPS, ROM_SIZE, N_FIELDS = len(OPS), 16, len(FIELDS)

### 參考實作 (reference emulator)

用 PyTorch 寫，一次可以模擬「一整批」CPU（在 GPU 上平行執行），用來產生訓練資料和驗證答案。

In [ ]:
def cpu_step(prog, s):
    """prog: (N, 16, 4), s: (N, 15)  ->  執行一個指令後的狀態 (N, 15)"""
    N = s.shape[0]
    ar = torch.arange(N, device=s.device)
    pc = s[:, 0]
    op, x, y, n = prog[ar, pc].unbind(-1)
    R, Z, CF, M = s[:, 1:5], s[:, 5], s[:, 6], s[:, 7:15]
    A, rx, ry = R[:, 0], R[ar, x], R[ar, y]
    newR, newZ, newCF, newM = R.clone(), Z.clone(), CF.clone(), M.clone()
    new_pc = (pc + 1) % ROM_SIZE

    def write(mask, idx, val, zero_flag=True):
        newR[ar[mask], idx[mask]] = val[mask]
        if zero_flag:
            newZ[mask] = (val[mask] == 0).long()

    is_ = lambda name: op == OP[name]
    write(is_("MOV"), x, ry, zero_flag=False)
    write(is_("MVI"), x, n, zero_flag=False)
    zeros = torch.zeros_like(A)
    for name, t in [("ADD", A + ry), ("SUB", A - ry)]:
        m = is_(name)
        write(m, zeros, t & 255)
        newCF[m] = ((t > 255) | (t < 0)).long()[m]
    write(is_("INC"), x, (rx + 1) & 255)
    write(is_("DEC"), x, (rx - 1) & 255)
    for name, v in [("AND", A & ry), ("OR", A | ry), ("XOR", A ^ ry)]:
        m = is_(name)
        write(m, zeros, v)
        newCF[m] = 0
    write(is_("LD"), x, M[ar, n % 8], zero_flag=False)
    m = is_("ST")
    newM[ar[m], (n % 8)[m]] = rx[m]
    jump = is_("JP") | (is_("JZ") & (Z == 1)) | (is_("JNZ") & (Z == 0)) | (is_("JC") & (CF == 1))
    new_pc = torch.where(jump, n % ROM_SIZE, new_pc)
    new_pc = torch.where(is_("HALT"), pc, new_pc)
    return torch.cat([new_pc[:, None], newR, newZ[:, None], newCF[:, None], newM], dim=1)

### 組譯器 (assembler) 與顯示工具

In [ ]:
def assemble(src):
    """把組合語言轉成 (16, 4) 的 ROM，不足的部份補 HALT"""
    rom = []
    for line in src.strip().split("\n"):
        line = line.split(";")[0].strip()
        if not line:
            continue
        name, *args = line.replace(",", " ").split()
        args = [a.strip("[]") for a in args]
        op, x, y, n = OP[name.upper()], 0, 0, 0
        reg = lambda a: REGS.index(a.upper())
        if name.upper() in ("MOV",):
            x, y = reg(args[0]), reg(args[1])
        elif name.upper() in ("MVI", "LD"):
            x, n = reg(args[0]), int(args[1])
        elif name.upper() == "ST":
            n, x = int(args[0]), reg(args[1])
        elif name.upper() in ("ADD", "SUB", "AND", "OR", "XOR"):
            y = reg(args[0])
        elif name.upper() in ("INC", "DEC"):
            x = reg(args[0])
        elif name.upper() in ("JP", "JZ", "JNZ", "JC"):
            n = int(args[0])
        rom.append([op, x, y, n])
    rom += [[OP["HALT"], 0, 0, 0]] * (ROM_SIZE - len(rom))
    return torch.tensor(rom)

def show_state(s):
    s = s.tolist()
    return (f"PC={s[0]:2d} A={s[1]:3d} B={s[2]:3d} C={s[3]:3d} D={s[4]:3d} Z={s[5]} CF={s[6]} "
            f"M={s[7:]}")

# 範例：用迴圈計算 6 x 7，結果存到 M0
MUL = """
    MVI A,0
    MVI B,7
    MVI C,6
    ADD B      ; 3: A += B
    DEC C
    JNZ 3
    ST [0],A
    HALT
"""
rom = assemble(MUL)
s = torch.zeros(1, N_FIELDS, dtype=torch.long)
for t in range(25):
    print(f"{t:2d} {OPS[rom[s[0,0], 0]]:5s}", show_state(s[0]))
    s = cpu_step(rom[None], s)

## 訓練資料：隨機程式 + 隨機狀態

下一個狀態只取決於「目前狀態」和「程式」，所以我們可以無限量地產生隨機的 (程式, 狀態) 配對，直接在 GPU 上用參考實作算出答案。

In [ ]:
def random_batch(N, device=device):
    op = torch.randint(0, N_OPS, (N, ROM_SIZE), device=device)
    x = torch.randint(0, 4, (N, ROM_SIZE), device=device)
    y = torch.randint(0, 4, (N, ROM_SIZE), device=device)
    n = torch.randint(0, 256, (N, ROM_SIZE), device=device)
    # 讓跳躍目標與記憶體位址大部份落在合理範圍
    small = torch.randint(0, 16, (N, ROM_SIZE), device=device)
    n = torch.where(torch.rand(N, ROM_SIZE, device=device) < 0.5, small, n)
    prog = torch.stack([op, x, y, n], dim=-1)
    s = torch.cat([torch.randint(0, ROM_SIZE, (N, 1), device=device),
                   torch.randint(0, 256, (N, 4), device=device),
                   torch.randint(0, 2, (N, 2), device=device),
                   torch.randint(0, 256, (N, 8), device=device)], dim=1)
    # 一部份的暫存器設成小數字、0、255，多一些進位和零的情況
    special = torch.tensor([0, 1, 2, 254, 255], device=device)
    mask = torch.rand(N, 4, device=device) < 0.2
    s[:, 1:5] = torch.where(mask, special[torch.randint(0, 5, (N, 4), device=device)], s[:, 1:5])
    return prog, s, cpu_step(prog, s)

prog, s, s_next = random_batch(3)
print(prog[0, :4])
print(s[0], "\n", s_next[0])

## 模型：Transformer 版的 CPU

輸入是 16 個「指令 token」+ 15 個「狀態 token」，共 31 個 token：

* **數字用二進位表示**：一個 byte 拆成 8 個 bit 再經過 `Linear` 變成向量；輸出也是預測 8 個 bit。這樣加法的進位規則比較容易學（比把 0~255 當成 256 個不相關的符號好學很多）
* 指令 token = opcode embedding + 暫存器 embedding + 立即值的 bit + **位址 embedding**（讓 attention 能用 PC 找到指令）
* 狀態 token = 數值的 bit + 欄位 embedding（這是 PC 還是 A 還是 M3…）
* 用雙向 (非 causal) 的 Transformer encoder：15 個狀態欄位的下一個值**同時**預測出來

In [ ]:
def to_bits(v):
    return ((v[..., None] >> torch.arange(8, device=v.device)) & 1).float()

def from_bits(logits):
    return ((logits > 0).long() << torch.arange(8, device=logits.device)).sum(-1)

class NeuralCPU(nn.Module):
    def __init__(self, d=256, n_layers=6, n_heads=8):
        super().__init__()
        self.op_emb = nn.Embedding(N_OPS, d)
        self.x_emb = nn.Embedding(4, d)
        self.y_emb = nn.Embedding(4, d)
        self.addr_emb = nn.Embedding(ROM_SIZE, d)
        self.field_emb = nn.Embedding(N_FIELDS, d)
        self.bits_in = nn.Linear(8, d)
        layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.norm = nn.LayerNorm(d)
        self.bits_out = nn.Linear(d, 8)

    def forward(self, prog, s):
        op, x, y, n = prog.unbind(-1)
        addr = torch.arange(ROM_SIZE, device=prog.device)
        fields = torch.arange(N_FIELDS, device=prog.device)
        prog_tok = self.op_emb(op) + self.x_emb(x) + self.y_emb(y) + self.bits_in(to_bits(n)) + self.addr_emb(addr)
        state_tok = self.bits_in(to_bits(s)) + self.field_emb(fields)
        h = self.encoder(torch.cat([prog_tok, state_tok], dim=1))
        return self.bits_out(self.norm(h[:, ROM_SIZE:]))       # (N, 15, 8) bit logits

    @torch.no_grad()
    def step(self, prog, s):
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            return from_bits(self(prog, s).float())

model = NeuralCPU().to(device)
print(f"參數數量: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 訓練

每一步都產生全新的 batch，所以不會 overfit。loss 是每個 bit 的 binary cross entropy。
T4 上大約 10 分鐘，可以先把 `TRAIN_STEPS` 改小試試看。

In [ ]:
@torch.no_grad()
def evaluate(model, N=20000):
    model.eval()
    prog, s, target = random_batch(N)
    pred = model.step(prog, s)
    ok = (pred == target).all(1)
    op = prog[torch.arange(N, device=device), s[:, 0], 0]
    per_op = {OPS[i]: ok[op == i].float().mean().item() for i in range(N_OPS)}
    return ok.float().mean().item(), per_op

def train(model, steps, batch_size=1024, lr=1e-3, log_every=500):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=steps, pct_start=0.05)
    scaler = torch.amp.GradScaler(device, enabled=(device == "cuda"))
    t0 = time.perf_counter()
    for step in range(1, steps + 1):
        model.train()
        prog, s, target = random_batch(batch_size)
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(prog, s)
        loss = F.binary_cross_entropy_with_logits(logits.float(), to_bits(target))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        if step % log_every == 0 or step == steps:
            acc, _ = evaluate(model, 4096)
            print(f"step {step:5d}  loss {loss.item():.5f}  單步完全正確率 {acc:.4f}  {time.perf_counter()-t0:.0f}s")

TRAIN_STEPS = 8000
train(model, TRAIN_STEPS)

## 評估一：單步準確率

隨機程式、隨機狀態，執行一個指令後，**15 個欄位全部正確**才算對。

In [ ]:
acc, per_op = evaluate(model, 50000)
print(f"整體單步正確率: {acc:.5f}  (錯誤率 {1-acc:.2e})")
for name, a in sorted(per_op.items(), key=lambda kv: kv[1]):
    print(f"  {name:5s} {a:.5f}")

## 評估二：真的跑程式

讓神經網路自己一步一步執行（每一步的輸出當作下一步的輸入，**不再給它正確答案**），和參考實作比較。

In [ ]:
def run(prog, s, steps, stepper):
    traj = [s]
    for _ in range(steps):
        s = stepper(prog, s)
        traj.append(s)
    return torch.stack(traj, dim=1)   # (N, steps+1, 15)

def compare(src, steps=100, name=""):
    rom = assemble(src)[None].to(device)
    s0 = torch.zeros(1, N_FIELDS, dtype=torch.long, device=device)
    ref = run(rom, s0, steps, cpu_step)[0]
    neu = run(rom, s0, steps, model.step)[0]
    diff = (ref != neu).any(1).nonzero()
    first = diff[0].item() if len(diff) else None
    print(f"{name}: 參考答案 {show_state(ref[-1])}")
    print(f"{' ' * len(name)}  神經CPU  {show_state(neu[-1])}")
    print("  ✅ 全部步驟一致" if first is None else f"  ❌ 第 {first} 步開始出錯")

FIB = """
    MVI A,1
    MVI B,0
    MVI D,10
    MOV C,A     ; 3
    ADD B
    MOV B,C
    DEC D
    JNZ 3
    ST [1],A
    HALT
"""
SUM = """
    MVI A,0
    MVI B,20
    ADD B       ; 2: A += B
    DEC B
    JNZ 2
    ST [2],A
    HALT
"""
compare(MUL, 40, "6x7")
compare(FIB, 60, "fib")
compare(SUM, 80, "1+..+20")

## 評估三：錯誤會累積

即使單步正確率有 99.9%，跑 $k$ 步都正確的機率大約是 $0.999^k$。真正的電腦每秒執行數百萬個指令，**任何一個 bit 錯了，程式就可能當掉**。

用 2000 個隨機程式各跑 100 步，看看「到第 k 步為止都完全正確」的比例：

In [ ]:
N, K = 2000, 100
prog, s0, _ = random_batch(N)
ref = run(prog, s0, K, cpu_step)
neu = run(prog, s0, K, model.step)
ok_until = (ref == neu).all(-1).cumprod(1).float().mean(0).cpu()
p1 = ok_until[1].item()
plt.plot(ok_until, label="neural CPU")
plt.plot([p1 ** k for k in range(K + 1)], "--", label=f"{p1:.4f}^k")
plt.xlabel("steps"); plt.ylabel("fraction still exactly correct"); plt.legend(); plt.grid(alpha=0.3)
plt.show()
print(f"100 步後仍完全正確的比例: {ok_until[-1]:.3f}")

## Q
自己寫一個程式（例如計算 $2^7$、比較兩個數字取最大值、計算 100 除以 7 的商），分別用參考實作和神經 CPU 執行並比較。

In [ ]:
# %load q_cpu_program.py
%run -i q_cpu_program.py

## Q
如果模型只有 **2 層**，哪些指令最難學？為什麼？（提示：8-bit 加法的進位可能要從第 0 個 bit 一路傳到第 7 個 bit）

In [ ]:
# %load q_cpu_small_model.py
%run -i q_cpu_small_model.py

## 評估四：速度——能跑紅白機嗎？

In [ ]:
rom = assemble(SUM)[None].to(device)
s = torch.zeros(1, N_FIELDS, dtype=torch.long, device=device)
model.step(rom, s)  # warm up
if device == "cuda": torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(200):
    s = model.step(rom, s)
if device == "cuda": torch.cuda.synchronize()
single = 200 / (time.perf_counter() - t0)

B = 8192
progB, sB, _ = random_batch(B)
model.step(progB, sB)
if device == "cuda": torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(20):
    sB = model.step(progB, sB)
if device == "cuda": torch.cuda.synchronize()
batched = 20 * B / (time.perf_counter() - t0)

nes = 1.79e6 / 3.5    # NES 的 6502：1.79 MHz，平均每個指令約 3.5 個 cycle
print(f"神經 CPU (單一台)   : {single:10,.0f} 指令/秒")
print(f"神經 CPU ({B} 台平行): {batched:10,.0f} 指令/秒 (總和)")
print(f"NES 的 6502         : {nes:10,.0f} 指令/秒")
print(f"單一台神經 CPU 比 NES 慢 {nes / single:,.0f} 倍；NES 一格畫面 (1/60 秒) 約 {nes/60:,.0f} 個指令，"
      f"神經 CPU 要 {nes / 60 / single:,.0f} 秒")

## 結論：能不能用 Transformer 跑 Z80、紅白機、Apple II？

| | TinyCPU (本單元) | Z80 / 6502 | 紅白機 (NES) / Apple II |
|---|---|---|---|
| 暫存器 | 4 個 8-bit | 7~14 個，含 16-bit | 6502：A, X, Y, SP, P, PC |
| 記憶體 | 8 bytes + 16 個指令，全部放進 context | 64KB 位址空間 | NES：2KB RAM + 卡匣 32KB+ + 影像晶片 PPU；Apple II：48KB RAM，畫面直接對應記憶體 |
| 指令數 | 17 | Z80 約 700 種 (含前綴)、6502 151 種 | 還要模擬 PPU、音效、時序、mapper |
| 可行性 | ✅ 單步可接近 100% | 單步「學」得起來，但記憶體放不進 context，要另外設計記憶體讀寫機制 | ❌ 速度慢上萬倍，而且錯一個 bit 遊戲就當掉 |

重點整理
* **單步** 的運算 Transformer 可以學得很好；困難在於**長時間精確執行**：錯誤會累積。這也是 LLM 做長串精確計算時會出錯的原因之一
* 所以 LLM 做精確計算時，通常讓它**呼叫工具**（例如寫 Python 程式去跑，見 04-LLM-Apps 的工具呼叫），而不是讓它「心算」
* 研究上也有人**手動設計權重**，讓 Transformer 精確地執行程式（例如 *Looped Transformers as Programmable Computers*, 2023；DeepMind 的 *Tracr* 可以把程式編譯成 Transformer 權重），這種做法不會出錯，但權重不是學出來的
* 如果目標是「看起來像在玩遊戲」，可以參考 **世界模型 (world model)**：例如 GameNGen (2024) 用擴散模型逐格生成 DOOM 的畫面、Oasis (2024) 生成 Minecraft。它們模仿的是畫面，而不是執行遊戲程式，訓練需要的算力也遠超過 T4